# Commercial data quotes and orders

See https://github.com/EO-DataHub/eodhp-guide/blob/main/examples/uc9-data-discovery-and-analysis/CommercialDataOrdering.ipynb
and Gemma, using the bottom bit of this notebook to get different orders for different licenses + clipped/unclipped

NEEDS REWORKING

In [ ]:
!pip install pystac-client xarray rasterio

# QUOTE DOC

## Storing and Loading Your API Key Securely

To securely store and use your API key in this notebook, follow these steps:

1. **Generate an API key**:
   - Follow instructions in the Getting Started documentation for Hub APIs to generate a Hub API Key.

2. **Create a `.env` File**:
   - In the same directory as this notebook, create a file named `.env`.
   - Add the following line to the file, replacing `<your_api_key>` with your actual API key:
     ```plaintext
     API_KEY=<your_api_key>
     ```
   - Save the file.

3. **Load the Key in the Notebook**:
   - The following code snippet is already included in this notebook to load the key securely:
     ```python
     import os
     from dotenv import load_dotenv
     
     load_dotenv(".env")  # Ensure the path matches your `.env` file location
     key = os.getenv("API_KEY")
     ```
   - This will load the `API_KEY` from your `.env` file into the `key` variable.

4. **Keep the `.env` File Secure**:
   - Do not share your `.env` file or API key.
   - If you do accidentally share your key, delete it in the workspaces UI and request a new one.

By following these steps, you can securely store and use your API key without exposing it in the notebook.

## Setting Up the Workspace and Environment

To ensure the notebook works correctly, you need to configure the `workspace` and `environment` variables. Follow these steps:
1. **Ensure your workspace is configured for commercial orders**
   - Follow instructions in the Hub documentation under `Data - Commercial` to link your commercial account to your workspace.
2. **Set the `workspace` Variable**:
   - In the following cell, the `workspace` variable should be set to the name of the workspace you wish to order data for.
   - You can find the workspace name in the site UI.
3. **Set the `environment` Variable**
   - In the following cell, The environment variable determines which environment you are working in. It can only be one of the following values:
     - prod (Production)
     - staging (Staging)
     - test (Testing)
     - dev (Development)

In [ ]:
import os
from dotenv import load_dotenv

workspace = "gemmaprodtest"
environment = "prod"

load_dotenv("gem-auth.txt")
api_key = os.getenv("API_KEY")

if environment == "prod":
    platform_domain = "https://eodatahub.org.uk"
elif environment == "staging":
    platform_domain = "https://staging.eodatahub.org.uk"
elif environment == "test":
    platform_domain = "https://test.eodatahub.org.uk"
elif environment == "dev":
    platform_domain = "https://dev.eodatahub.org.uk"
else:
    raise ValueError("Invalid environment. Choose from 'prod', 'staging', 'test', or 'dev'.")

# Create a pystac-client Client for the EODH Catalogue

The EODH catalogue can be viewed through a UI on the Hub under the `Catalogue` tab.

Our STAC catalogue can also be browsed programatically using tools such as pystac-client, using your api_key for authorisation. In the following example, our client is scoped to the Planet catalogue to refine a search for some Planet commercial data.

In [ ]:
from pystac_client import Client

# Limit scope of the top-level catalogue to Planet
rc_url = f"{platform_domain}/api/catalogue/stac/catalogs/commercial/catalogs/airbus"

# Create STAC client
stac_client = Client.open(rc_url, headers={"Authorization": f"Bearer {api_key}"})

In [ ]:
geom = {
    "type": "Polygon",
    "coordinates": [
        [
            [9.6, 57.1],
            [9.6, 57.0],
            [9.8, 56.9],
            [9.8, 57.0],
            [9.6, 57.1],
        ]
    ],
}
search = stac_client.search(
    max_items=10,
    collections=['PSScene'],
    intersects=geom,
)
for item in search.items():
    print(item.get_self_href())

In [ ]:
import requests

item_href = "https://staging.eodatahub.org.uk/api/catalogue/stac/catalogs/commercial/catalogs/planet/collections/PSScene/items/20250217_101155_07_24c7"
url = f"{item_href}/quote"
headers = {
    'accept': 'application/json', 
    'Content-Type': 'application/json', 
    'Authorization': f'Bearer {api_key}'
}
data =  {
    "coordinates": [
        [
            [9.6, 57.1],
            [9.6, 57.0],
            [9.8, 56.9],
            [9.8, 57.0],
            [9.6, 57.1]
        ]
    ]
}

response = requests.post(url, headers=headers, json=data)

print("Status Code", response.status_code)
print("Response ", response.json())

In [ ]:
import requests

item_href = "https://staging.eodatahub.org.uk/api/catalogue/stac/catalogs/commercial/catalogs/planet/collections/PSScene/items/20250217_101155_07_24c7"
url = f"{item_href}/order"
headers = {
    'accept': 'application/json', 
    'Content-Type': 'application/json', 
    'Authorization': f'Bearer {api_key}'
}
data =  {
    "productBundle": "General use",
    "coordinates": [
        [
            [9.6, 57.1],
            [9.6, 57.0],
            [9.8, 56.9],
            [9.8, 57.0],
            [9.6, 57.1]
        ]
    ]
}

response = requests.post(url, headers=headers, json=data)

print("Status Code:", response.status_code)
print("Response:", response.json())

location_header = response.headers.get('Location')
print("Item will shortly be available at:", location_header)

In [ ]:
from pystac import Item
from pystac_client import CollectionClient

data_i_ordered_earlier = f"{platform_domain}/api/catalogue/stac/catalogs/user/catalogs/{workspace}/catalogs/commercial-data/catalogs/planet"
stac_client = Client.open(data_i_ordered_earlier, headers={"Authorization": f"Bearer {api_key}"})

In [ ]:
ordered_item = next(stac_client.get_items("20250217_101155_07_24c7"))
ordered_item

In [ ]:
asset = ordered_item.get_assets()["20250217_101155_07_24c7_3B_AnalyticMS_clip.tif"]
asset

In [ ]:
import urllib3
from io import BytesIO

resp = urllib3.request("GET", asset.href, headers={"Authorization": f"Bearer {api_key}"})
resp.data[0:100]

# Obtain a Quote for Airbus Optical Data

Search and ordering capabilities are also in place for Airbus Optical items. Once an item of interest is found, you may obtain a quote for the item via a POST request to /quote following the item href.

Change the `item_href` to any valid link to a STAC item in our Airbus commercial catalogue to browse prices. Change the `coordinates` to an AOI to clip the item if required.

Note: Coordinates are to be provided in longitude/latitude WGS84, and the polygon must be closed. Coordinates follow OGC GeoJSON specification.

Change the `licence` to another supported licence if required. Supported licences are:
  - `Standard`
  - `Background Layer`
  - `Standard + Background Layer`
  - `Academic`
  - `Media Licence`
  - `Standard Multi End-Users (2-5)`
  - `Standard Multi End-Users (6-10)`
  - `Standard Multi End-Users (11-30)`
  - `Standard Multi End-Users (>30)`

In [ ]:
# Tower Hamlets BBOX
[
    [-0.08017620447063, 51.4845636802758], 
    [0.009932571058582, 51.4845636802758], 
    [0.009932571058582, 51.5446504200296], 
    [-0.08017620447063, 51.5446504200296], 
    [-0.08017620447063, 51.4845636802758]
]
# Camden BBOX
[
    [-0.213437487930831, 51.5126101149581], 
    [-0.105323965744685, 51.5126101149581], 
    [-0.105323965744685, 51.572981384291], 
    [-0.213437487930831, 51.572981384291], 
    [-0.213437487930831, 51.5126101149581]
]
# Brent BBOX
[
    [-0.335562160691732, 51.5276559502351], 
    [-0.19145849063202, 51.5276559502351], 
    [-0.19145849063202, 51.6003731988738], 
    [-0.335562160691732, 51.6003731988738], 
    [-0.335562160691732, 51.5276559502351]
]

In [ ]:
collection_id = "airbus_phr_data"
img_id = "ACQ_PNEO4_03813510086486"

import requests

item_href = f'https://eodatahub.org.uk/api/catalogue/stac/catalogs/commercial/catalogs/airbus/collections/airbus_pneo_data/items/{img_id}'
url = f"{item_href}/quote" #<--- specifies quote
headers = {
    'accept': 'application/json', 
    'Content-Type': 'application/json', 
    'Authorization': f'Bearer {api_key}'
}
data =  {
    "licence": "Standard Multi End-Users (2-5)",
    #"licence": "Standard Multi End-Users (6-10)",

}

response = requests.post(url, headers=headers, json=data)

print("Status Code", response.status_code)
print("Response ", response.json())

## Setup - Load an API Key and Libraries

## Storing and Loading Your API Key Securely

To securely store and use your API key in this notebook, follow these steps:

1. **Generate an API key**:
   - Follow instructions in the Getting Started documentation for Hub APIs to generate a Hub API Key.

2. **Create a `.env` File**:
   - In the same directory as this notebook, create a file named `.env`.
   - Add the following line to the file, replacing `<your_api_key>` with your actual API key:
     ```plaintext
     API_KEY=<your_api_key>
     ```
   - Save the file.

3. **Load the Key in the Notebook**:
   - The following code snippet is already included in this notebook to load the key securely:
     ```python
     import os
     from dotenv import load_dotenv
     
     load_dotenv(".env")  # Ensure the path matches your `.env` file location
     key = os.getenv("API_KEY")
     ```
   - This will load the `API_KEY` from your `.env` file into the `key` variable.

4. **Keep the `.env` File Secure**:
   - Do not share your `.env` file or API key.
   - If you do accidentally share your key, delete it in the workspaces UI and request a new one.

By following these steps, you can securely store and use your API key without exposing it in the notebook.

## Setting Up the Workspace and Environment

To ensure the notebook works correctly, you need to configure the `workspace` and `environment` variables. Follow these steps:
1. **Ensure your workspace is configured for commercial orders**
   - Follow instructions in the Hub documentation under `Data - Commercial` to link your commercial account to your workspace.
2. **Set the `workspace` Variable**:
   - In the following cell, the `workspace` variable should be set to the name of the workspace you wish to order data for.
   - You can find the workspace name in the site UI.
3. **Set the `environment` Variable**
   - In the following cell, The environment variable determines which environment you are working in. It can only be one of the following values:
     - prod (Production)
     - staging (Staging)
     - test (Testing)
     - dev (Development)

In [1]:
import os
from dotenv import load_dotenv

#workspace = "airbus-nceo-quota"
workspace = "imago-temp"
environment = "prod"

#load_dotenv("airbus-nceo-quota-auth.txt")
#api_key = os.getenv("API_KEY")
#api_key = "454f4448086849a118dfc9c77aadde80"
api_key = "454f4448af5a97fceaa09cabeaa15b7f"
print(api_key)

if environment == "prod":
    platform_domain = "https://eodatahub.org.uk"
elif environment == "staging":
    platform_domain = "https://staging.eodatahub.org.uk"
elif environment == "test":
    platform_domain = "https://test.eodatahub.org.uk"
elif environment == "dev":
    platform_domain = "https://dev.eodatahub.org.uk"
else:
    raise ValueError("Invalid environment. Choose from 'prod', 'staging', 'test', or 'dev'.")

ModuleNotFoundError: No module named 'dotenv'

In [ ]:
!pip install pystac-client xarray rasterio

# Create a pystac-client Client for the EODH Catalogue

The EODH catalogue can be viewed through a UI on the Hub under the `Catalogue` tab.

Our STAC catalogue can also be browsed programatically using tools such as pystac-client, using your api_key for authorisation. In the following example, our client is scoped to the Planet catalogue to refine a search for some Planet commercial data.

In [54]:
from pystac_client import Client

# Limit scope of the top-level catalogue to Planet
rc_url = f"{platform_domain}/api/catalogue/stac/catalogs/commercial/catalogs/airbus"

# Create STAC client
stac_client = Client.open(rc_url, headers={"Authorization": f"Bearer {api_key}"})

# Obtain a Quote for Airbus Optical Data

Search and ordering capabilities are also in place for Airbus Optical items. Once an item of interest is found, you may obtain a quote for the item via a POST request to /quote following the item href.

Change the `item_href` to any valid link to a STAC item in our Airbus commercial catalogue to browse prices. Change the `coordinates` to an AOI to clip the item if required.

Note: Coordinates are to be provided in longitude/latitude WGS84, and the polygon must be closed. Coordinates follow OGC GeoJSON specification.

Change the `licence` to another supported licence if required. Supported licences are:
  - `Standard`
  - `Background Layer`
  - `Standard + Background Layer`
  - `Academic`
  - `Media Licence`
  - `Standard Multi End-Users (2-5)`
  - `Standard Multi End-Users (6-10)`
  - `Standard Multi End-Users (11-30)`
  - `Standard Multi End-Users (>30)`

In [4]:
# Tower Hamlets BBOX
[
    [-0.08017620447063, 51.4845636802758], 
    [0.009932571058582, 51.4845636802758], 
    [0.009932571058582, 51.5446504200296], 
    [-0.08017620447063, 51.5446504200296], 
    [-0.08017620447063, 51.4845636802758]
]
# Camden BBOX
[
    [-0.213437487930831, 51.5126101149581], 
    [-0.105323965744685, 51.5126101149581], 
    [-0.105323965744685, 51.572981384291], 
    [-0.213437487930831, 51.572981384291], 
    [-0.213437487930831, 51.5126101149581]
]
# Brent BBOX
[
    [-0.335562160691732, 51.5276559502351], 
    [-0.19145849063202, 51.5276559502351], 
    [-0.19145849063202, 51.6003731988738], 
    [-0.335562160691732, 51.6003731988738], 
    [-0.335562160691732, 51.5276559502351]
]

[[-0.335562160691732, 51.5276559502351],
 [-0.19145849063202, 51.5276559502351],
 [-0.19145849063202, 51.6003731988738],
 [-0.335562160691732, 51.6003731988738],
 [-0.335562160691732, 51.5276559502351]]

In [5]:
# Nottingham BBOX
[
    [-1.200017602930003, 52.916903709974115], 
    [-1.115113721172645, 52.916903709974115], 
    [-1.115113721172645, 52.96421445906777], 
    [-1.200017602930003, 52.96421445906777], 
    [-1.200017602930003, 52.916903709974115]
]

# Newark BBOX
[
    [-0.881986455899147, 53.04026682559284], 
    [-0.806024704932811, 53.04026682559284], 
    [-0.806024704932811, 53.0867959084634], 
    [-0.881986455899147, 53.0867959084634], 
    [-0.881986455899147, 53.04026682559284]
]

[[-0.881986455899147, 53.04026682559284],
 [-0.806024704932811, 53.04026682559284],
 [-0.806024704932811, 53.0867959084634],
 [-0.881986455899147, 53.0867959084634],
 [-0.881986455899147, 53.04026682559284]]

In [6]:
# Connahs Quay
[
    [-3.125615283937652, 53.2151149280873], 
    [-3.041636563289444, 53.2151149280873], 
    [-3.041636563289444, 53.250615760211225], 
    [-3.125615283937652, 53.250615760211225], 
    [-3.125615283937652, 53.2151149280873]
]

# Llanwern solar farm
[
    [-2.928017830575294, 51.53144166954083], 
    [-2.846198895128091, 51.53144166954083], 
    [-2.846198895128091, 51.58599439873845], 
    [-2.928017830575294, 51.58599439873845], 
    [-2.928017830575294, 51.53144166954083]
]

[[-2.928017830575294, 51.53144166954083],
 [-2.846198895128091, 51.53144166954083],
 [-2.846198895128091, 51.58599439873845],
 [-2.928017830575294, 51.58599439873845],
 [-2.928017830575294, 51.53144166954083]]

In [7]:
# Historic England - Yorkshire Wolds
[
    [-0.7194120178713573, 54.019530606882086], 
    [-0.6551900586344416, 54.019530606882086], 
    [-0.6551900586344416, 54.119033049118705], 
    [-0.7194120178713573, 54.119033049118705], 
    [-0.7194120178713573, 54.019530606882086]
]

[[-0.7194120178713573, 54.019530606882086],
 [-0.6551900586344416, 54.019530606882086],
 [-0.6551900586344416, 54.119033049118705],
 [-0.7194120178713573, 54.119033049118705],
 [-0.7194120178713573, 54.019530606882086]]

In [31]:
# Durham
[
    [-1.646347957199817, 54.74952384519631], 
    [-1.528779162828712, 54.74952384519631], 
    [-1.528779162828712, 54.79899159582869], 
    [-1.646347957199817, 54.79899159582869], 
    [-1.646347957199817, 54.74952384519631]
]
# Liverpool
[
    [-3.023027015997294, 53.427185806245404], 
 [-2.735933462471953, 53.427185806245404], 
 [-2.735933462471953, 53.570968176022525], 
 [-3.023027015997294, 53.570968176022525], 
 [-3.023027015997294, 53.427185806245404]
]
# Aberdeen
[
 [-2.198833134075246, 57.11270465250282], 
 [-2.039788856021223, 57.11270465250282], 
 [-2.039788856021223, 57.19006479564655], 
 [-2.198833134075246, 57.19006479564655], 
 [-2.198833134075246, 57.11270465250282]
]
# Edinburgh
[
    [-3.262756419837451, 55.91305330462832], 
    [-3.107444674670536, 55.91305330462832], 
    [-3.107444674670536, 55.96997598985442], 
    [-3.262756419837451, 55.96997598985442], 
    [-3.262756419837451, 55.91305330462832]
]
# Wales
[
    [-3.241168606148315, 51.76342931732691], 
    [-3.156278683786832, 51.76342931732691], 
    [-3.156278683786832, 51.79567191596545], 
    [-3.241168606148315, 51.79567191596545], 
    [-3.241168606148315, 51.76342931732691]
]
# Wales 2
[
    [-3.512843316498004, 51.52240965774117], 
    [-3.283665874657831, 51.52240965774117], 
    [-3.283665874657831, 51.71108459160826], 
    [-3.512843316498004, 51.71108459160826], 
    [-3.512843316498004, 51.52240965774117]
]

[[-3.262756419837451, 55.91305330462832],
 [-3.107444674670536, 55.91305330462832],
 [-3.107444674670536, 55.96997598985442],
 [-3.262756419837451, 55.96997598985442],
 [-3.262756419837451, 55.91305330462832]]

In [ ]:
# Rothera BAS research station
[
    [-68.23932240331028, -67.5891691014489], 
    [-68.0617356765442, -67.5891691014489], 
    [-68.0617356765442, -67.52992345141907], 
    [-68.23932240331028, -67.52992345141907], 
    [-68.23932240331028, -67.5891691014489]
]

### Backup code

In [6]:
collection_id = "airbus_pneo_data"

In [37]:
img_id = "ACQ_PNEO3_05619416392111"

In [38]:
import requests

item_href = f'https://eodatahub.org.uk/api/catalogue/stac/catalogs/commercial/catalogs/airbus/collections/airbus_pneo_data/items/{img_id}'
url = f"{item_href}/quote" #<--- specifies quote
headers = {
    'accept': 'application/json', 
    'Content-Type': 'application/json', 
    'Authorization': f'Bearer {api_key}'
}
data =  {
    "licence": "Standard Multi End-Users (2-5)",
}

response = requests.post(url, headers=headers, json=data)

print("Status Code", response.status_code)
print("Response ", response.json())

Status Code 200
Response  {'value': 13176.0, 'units': 'EUR'}


In [39]:
import requests

item_href = f'https://eodatahub.org.uk/api/catalogue/stac/catalogs/commercial/catalogs/airbus/collections/airbus_pneo_data/items/{img_id}'
url = f"{item_href}/quote" #<--- specifies quote
headers = {
    'accept': 'application/json', 
    'Content-Type': 'application/json', 
    'Authorization': f'Bearer {api_key}'
}
data =  {
    "licence": "Standard Multi End-Users (6-10)",
}

response = requests.post(url, headers=headers, json=data)

print("Status Code", response.status_code)
print("Response ", response.json())

Status Code 200
Response  {'value': 15811.2, 'units': 'EUR'}


In [40]:
import requests

item_href = f'https://eodatahub.org.uk/api/catalogue/stac/catalogs/commercial/catalogs/airbus/collections/airbus_pneo_data/items/{img_id}'
url = f"{item_href}/quote" #<--- specifies quote
headers = {
    'accept': 'application/json', 
    'Content-Type': 'application/json', 
    'Authorization': f'Bearer {api_key}'
}
data =  {
    "licence": "Standard Multi End-Users (2-5)",
    "coordinates": [
        [
            [-1.200017602930003, 52.916903709974115], 
            [-1.115113721172645, 52.916903709974115], 
            [-1.115113721172645, 52.96421445906777], 
            [-1.200017602930003, 52.96421445906777], 
            [-1.200017602930003, 52.916903709974115]
        ]
    ]
}

response = requests.post(url, headers=headers, json=data)

print("Status Code", response.status_code)
print("Response ", response.json())

Status Code 200
Response  {'value': 360.0, 'units': 'EUR'}


In [41]:
import requests

item_href = f'https://eodatahub.org.uk/api/catalogue/stac/catalogs/commercial/catalogs/airbus/collections/airbus_pneo_data/items/{img_id}'
url = f"{item_href}/quote" #<--- specifies quote
headers = {
    'accept': 'application/json', 
    'Content-Type': 'application/json', 
    'Authorization': f'Bearer {api_key}'
}
data =  {
    "licence": "Standard Multi End-Users (6-10)",
    "coordinates": [
        [
            [-1.200017602930003, 52.916903709974115], 
            [-1.115113721172645, 52.916903709974115], 
            [-1.115113721172645, 52.96421445906777], 
            [-1.200017602930003, 52.96421445906777], 
            [-1.200017602930003, 52.916903709974115]
        ]
    ]
}

response = requests.post(url, headers=headers, json=data)

print("Status Code", response.status_code)
print("Response ", response.json())

Status Code 200
Response  {'value': 432.0, 'units': 'EUR'}


## Place order 2-5 license clipped
### Run this code block to order

In [63]:
img_id = "ACQ_PNEO4_06029801689587"

In [64]:
import requests

item_href = f'https://eodatahub.org.uk/api/catalogue/stac/catalogs/commercial/catalogs/airbus/collections/airbus_pneo_data/items/{img_id}'
url = f"{item_href}/quote" #<--- specifies quote
headers = {
    'accept': 'application/json', 
    'Content-Type': 'application/json', 
    'Authorization': f'Bearer {api_key}'
}
data =  {
    "licence": "Standard Multi End-Users (2-5)",
    "coordinates": [
        [
            [-3.512843316498004, 51.52240965774117], 
            [-3.283665874657831, 51.52240965774117], 
            [-3.283665874657831, 51.71108459160826], 
            [-3.512843316498004, 51.71108459160826], 
            [-3.512843316498004, 51.52240965774117]
        ]
    ]
}

response = requests.post(url, headers=headers, json=data)

print("Status Code", response.status_code)
print("Response ", response.json())

Status Code 200
Response  {'value': 4348.8, 'units': 'EUR'}


In [62]:
import requests

item_href = f'https://eodatahub.org.uk/api/catalogue/stac/catalogs/commercial/catalogs/airbus/collections/airbus_pneo_data/items/{img_id}'
url = f"{item_href}/order" #<--- places the order
headers = {
    'accept': 'application/json', 
    'Content-Type': 'application/json', 
    'Authorization': f'Bearer {api_key}'
}
data =  {
    "productBundle": "Analytic", # "General Use", "Visual" 'Basic' or 'Analytic'
    "licence": "Standard Multi End-Users (2-5)",
    "endUserCountry": "GB",
    "coordinates": [
        [
         [-2.198833134075246, 57.11270465250282], 
         [-2.039788856021223, 57.11270465250282], 
         [-2.039788856021223, 57.19006479564655], 
         [-2.198833134075246, 57.19006479564655], 
         [-2.198833134075246, 57.11270465250282]
        ]
    ]
}

#response = requests.post(url, headers=headers, json=data)

print("Status Code", response.status_code)
print("Response ", response.json())

location_header = response.headers.get('Location')
print("Item will shortly be available at:", location_header)

Status Code 201
Response  {'type': 'Feature', 'stac_version': '1.0.0', 'stac_extensions': ['https://stac-extensions.github.io/eo/v2.0.0/schema.json', 'https://stac-extensions.github.io/view/v1.0.0/schema.json', 'https://stac-extensions.github.io/order/v1.1.0/schema.json'], 'id': 'ACQ_PNEO4_05712000144083_Analytic-f9b5bbc5467b090b4de1f406857bb338', 'collection': 'airbus_pneo_data', 'geometry': {'type': 'Polygon', 'coordinates': [[[-2.1007699918, 57.138207196], [-2.101026704632496, 57.11270465250282], [-2.198833134075246, 57.11270465250282], [-2.198833134075246, 57.136710680441105], [-2.1007699918, 57.138207196]]]}, 'bbox': [-2.4092407644, 56.9970115419, -2.1007699918, 57.138207196], 'properties': {'datetime': '2025-05-14T11:06:41.577000Z', 'updated': '2025-12-19T15:56:09.271301Z', 'platform': 'PNEO4', 'constellation': 'PNEO', 'gsd': 0.3, 'access': ['HTTPServer'], 'eo:cloud_cover': 0, 'eo:snow_cover': 0, 'view:azimuth': 180.0140170237114, 'view:sun_azimuth': 158.26276494514673, 'view:sun